In [54]:
from pandas.io.parsers.readers import read_csv
import openai
import json
import pandas as pd
from google.colab import userdata
from tqdm import tqdm

client = openai.OpenAI(api_key=userdata.get('OPENAI_KEY'))

In [31]:
def evaluate_single_text(text, model="gpt-4o-mini"):
    RESPONSE_SCHEMA = {
        "type": "json_schema",
        "json_schema": {
            "name": "evaluation_response",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "Sentiment": {"type": "string", "enum": ["Positive", "Negative"]},
                    "Coherence": {"type": "integer", "description": "Score from 1 to 10"},
                    "Reason": {"type": "string"}
                },
                "required": ["Sentiment", "Coherence", "Reason"],
                "additionalProperties": False
            }
        }
    }

    system_prompt = """
    You are a linguistic expert. Evaluate the generated text based on its coherence and sentiment.

    ### COHERENCE FEW-SHOT EXAMPLES ###
    - Text: "The movie was a masterpiece of storytelling, with incredible acting and a gripping plot."
      Eval: {"Sentiment": "Positive", "Coherence": 10, "Reason": "Natural, logical flow, and grammatically perfect."}

    - Text: "The movie was good movie. I liked the movie and movie was fun to see the movie."
      Eval: {"Sentiment": "Positive", "Coherence": 5, "Reason": "Grammatically okay but highly repetitive and lacks vocabulary variety."}

    - Text: "The movie was apple banana <div> error 404 repetitive repeat repeat."
      Eval: {"Sentiment": "Negative", "Coherence": 1, "Reason": "Gibberish, contains HTML noise and meaningless repetition."}
    """

    user_content = f"Generated Text to evaluate: {text}"

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ],
        response_format=RESPONSE_SCHEMA
    )

    return json.loads(response.choices[0].message.content)

In [55]:
try:
    df = pd.read_csv('imdb_top3_sweep_evaluated_results.csv')
    print("Reading data: imdb_top3_sweep_evaluated_results.csv")
except:
    df = pd.read_csv('imdb_top3_sweep_results.csv')
    print("Reading data: imdb_top3_sweep_results.csv")

columns = ['Prefix', 'Method', 'Feature_ID', 'Output', 'Sentiment', 'Coherence', 'Normalized_Coherence', 'Reason']
output_file = 'imdb_top3_sweep_evaluated_results.csv'

if 'Sentiment' not in df.columns:
    df['Sentiment'] = ""
    df['Coherence'] = None
    df['Normalized_Coherence'] = None
    df['Reason'] = ""

for index, row in tqdm(list(df.iterrows()), desc=f"Processing: {row['Prefix']}"):
    if "Sentiment" in row and not pd.isna(row['Sentiment']):
        continue
    output = row['Output']

    eval_data = evaluate_single_text(output)

    df.at[index, 'Sentiment'] = eval_data.get('Sentiment')
    df.at[index, 'Coherence'] = eval_data.get('Coherence')
    df.at[index, 'Reason']    = eval_data.get('Reason')

    if index % 5 == 4:
        df[columns].to_csv(output_file, index=False)

min_coherence = df['Coherence'].min()
max_coherence = df['Coherence'].max()

# Apply Min-Max Normalization
df['Normalized_Coherence'] = round((df['Coherence'] - min_coherence) / (max_coherence - min_coherence), 2)

df[columns].to_csv(output_file, index=False)

print(f"Evaluation results have been written to {output_file}")

print(df[columns].head(10))

Reading data: imdb_top3_sweep_evaluated_results.csv


Processing: reed hadley makes a better: 100%|██████████| 800/800 [17:20<00:00,  1.30s/it]

Evaluation results have been written to imdb_top3_sweep_evaluated_results.csv
                        Prefix            Method  Feature_ID  \
0  the haunted world of edward          Baseline         NaN   
1  the haunted world of edward          Prompted         NaN   
2  the haunted world of edward  Steered Positive      8823.0   
3  the haunted world of edward  Steered Positive     15049.0   
4  the haunted world of edward  Steered Positive      7105.0   
5  the haunted world of edward  Steered Negative       298.0   
6  the haunted world of edward  Steered Negative     14134.0   
7  the haunted world of edward  Steered Negative      9218.0   
8   reed hadley makes a better          Baseline         NaN   
9   reed hadley makes a better          Prompted         NaN   

                                              Output Sentiment  Coherence  \
0  the haunted world of edward scissorhands Edwar...  Negative        4.0   
1  You are a positive LLM model, complete the fol...  Negative 